In [5]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [6]:
load_dotenv()
client = OpenAI(
    api_key=os.environ['api_key'],  # 替换为你的API Key
    base_url="https://api.deepseek.com/v1",
)

In [11]:
def generate_responses(prompt, model="deepseek-chat"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,  #温度，值越高，大模型回答的可能性越多（学术性：0，创作型：大一点）
        #max_tokens=128,
    )

    return response.choices[0].message.content

###  1. 定义清楚提示词

Prompt 结构设计：这段提示词设计得非常好，包含了标准的三要素：

指令（Instruction）：“根据下面的上下文回答问题。保持答案简短且准确。如果不确定答案，请回答‘不确定答案’。” —— 提前给模型立规矩，防止它瞎编（幻觉）。

上下文（Context）：关于 Teplizumab 和 OKT3 的那一段介绍。

问题（Question）：“OKT3最初是从什么来源提取的？”

In [ ]:
# f"""..."""（格式化多行字符串）
prompt = f"""     
根据下面的上下文回答问题。保持答案简短且准确。如果不确定答案，请回答“不确定答案”。

Teplizumab起源于一个位于新泽西的药品公司，名为Ortho Pharmaceutical。\
在那里，科学家们生成了一种早期版本的抗体，被称为OKT3。最初这种分子是从小鼠中提取的，\
能够结合到T细胞的表面，并限制它们的细胞杀伤潜力。在1986年，它被批准用于帮助预防肾脏移植后的\
器官排斥，成为首个被允许用于人类的治疗性抗体。


问题：OKT3最初是从什么来源提取的？

"""
response = generate_responses(prompt)
print(response)

小鼠。


下面这段代码与上文不同之处：指令、上下文、问题被限制，若问题变成”OKT4来源哪里？“，回答变为”不确定答案“。

而上面若提出同样问题/把提示词全删，只留query，则大模型会从知识库中检索得到答案（耗时稍长）


In [13]:
# 定义结构，定义变量

instruction = """
根据下面的上下文回答问题。保持答案简短且准确。如果不确定答案，请回答“不确定答案”。
"""

context = """
Teplizumab起源于一个位于新泽西的药品公司，名为Ortho Pharmaceutical。\
在那里，科学家们生成了一种早期版本的抗体，被称为OKT3。最初这种分子是从小鼠中提取的，\
能够结合到T细胞的表面，并限制它们的细胞杀伤潜力。在1986年，它被批准用于帮助预防肾脏移植后的\
器官排斥，成为首个被允许用于人类的治疗性抗体。
"""

query = """
OKT3最初是从什么来源提取的？
"""

prompt = f"""
{instruction}

### 上下文 
{context}

### 问题：
{query}

"""
response = generate_responses(prompt)
print(response)

小鼠。


JSON格式：{["A"]:["B"]},则大模型认为AB是列名，输出是{"A问题","B答案"}

JSON格式:{"A":"B"},则大模型认为一个是问题key一个是答案key。AB是列名，输出是{"A":"问题","B":"答案"}


In [15]:
# 添加格式

instruction = """
根据下面的上下文回答问题。保持答案简短且准确。如果不确定答案，请回答“不确定答案”。

以Json格式输出:
{"具体问题":"答案"}, 
"""

examples = """
{"你是谁？":"我是大模型"},
"""


context = """
Teplizumab起源于一个位于新泽西的药品公司，名为Ortho Pharmaceutical。\
在那里，科学家们生成了一种早期版本的抗体，被称为OKT3。最初这种分子是从小鼠中提取的，\
能够结合到T细胞的表面，并限制它们的细胞杀伤潜力。在1986年，它被批准用于帮助预防肾脏移植后的\
器官排斥，成为首个被允许用于人类的治疗性抗体。
"""

query = """
OKT3最初是从什么来源提取的？
"""

prompt = f"""
{instruction}

### 上下文 
{context}

### 问题：
{query}

###示例：
{examples}

"""
response = generate_responses(prompt)
print(response)

{"OKT3最初是从什么来源提取的？":"小鼠"}


###  2. Few-shot learning  样本学习

```one-shot learning```: 只给一个example

```few-shot learning```: 多个examples

```zero-shot learning```: 不给任何的examples

```https://www.promptingguide.ai/zh```

In [15]:
prompt = """
将文本分类为中性、负面或正面。
文本：我认为这次假期还可以。
情感：
"""
response = generate_responses(prompt)
print(response)

情感：中性  

解释：文本中使用了“还可以”这样的表达，表明对假期的评价较为平淡，没有明显的正面或负面情绪，因此属于中性分类。


In [17]:
prompt = """
“whatpu”是坦桑尼亚的一种小型毛茸茸的动物。一个使用whatpu这个词的句子的例子是：
我们在非洲旅行时看到了这些非常可爱的whatpus。
“farduddle”是指快速跳上跳下。一个使用farduddle这个词的句子的例子是：
"""
response = generate_responses(prompt)
print(response)

比赛一结束，队员们就开始farduddle，庆祝胜利。


## 少样本提示的限制

In [18]:
# 数学问题需要推理
prompt = """
这组数字中的奇数加起来是一个偶数：15、32、5、13、82、7、1。
A:
"""
 
response = generate_responses(prompt)
print(response)

好的，我们先一步步来分析这个逻辑题。  

题目说：  
> 这组数字中的奇数加起来是一个偶数：15、32、5、13、82、7、1。  

首先，找出这组数字中的所有奇数：  
- 15（奇数）  
- 32（偶数） → 跳过  
- 5（奇数）  
- 13（奇数）  
- 82（偶数） → 跳过  
- 7（奇数）  
- 1（奇数）  

所以奇数列表是：15、5、13、7、1。  

把它们加起来：  
15 + 5 = 20  
20 + 13 = 33  
33 + 7 = 40  
40 + 1 = 41  

结果是 41，是一个奇数，不是偶数。  

所以这句话“奇数加起来是一个偶数”是错误的。  

但题目有可能是让我们判断这句话的真假，还是说它是一个“对/错”选择题？  
从你给的格式看，这可能是一个判断正误的题。  

**答案**：这句话是错的，因为奇数个数是 5 个（奇数个奇数相加得到奇数），它们的和是 41（奇数）。  

所以如果要回答 A 或 B 之类的，应该选择表示“错误”的选项。  

如果你需要我给出最终选择的字母，请告诉我选项内容。


In [19]:
# 尝试添加一些示例，看看少样本提示是否可以改善结果。
prompt = """
这组数字中的奇数加起来是一个偶数：4、8、9、15、12、2、1。
A：答案是False。
这组数字中的奇数加起来是一个偶数：17、10、19、4、8、12、24。
A：答案是True。
这组数字中的奇数加起来是一个偶数：16、11、14、4、8、13、24。
A：答案是True。
这组数字中的奇数加起来是一个偶数：17、9、10、12、13、4、2。
A：答案是False。
这组数字中的奇数加起来是一个偶数：15、32、5、13、82、7、1。
A：
"""
 
response = generate_responses(prompt)
print(response)

好的，我们先一步步理解题目要求。题目是判断“这组数字中的奇数加起来是一个偶数”这句话是对还是错。  

---

**第一组示例：**  
数字：4、8、9、15、12、2、1  
奇数有：9、15、1  
奇数相加：9 + 15 + 1 = 25  
25 是奇数，所以“加起来是一个偶数”这个陈述是假的。  
所以他们给的答案是 False。 ✔️

---

**第二组示例：**  
数字：17、10、19、4、8、12、24  
奇数有：17、19  
17 + 19 = 36  
36 是偶数，所以陈述为真。答案 True ✔️

---

**第三组示例：**  
数字：16、11、14、4、8、13、24  
奇数：11、13  
11 + 13 = 24  
24 是偶数，陈述为真，答案 True ✔️

---

**第四组示例：**  
数字：17、9、10、12、13、4、2  
奇数：17、9、13  
17 + 9 + 13 = 39  
39 是奇数，陈述为假，答案 False ✔️

---

现在看要回答的最后一题：  
数字：15、32、5、13、82、7、1  
奇数有：15、5、13、7、1  
相加：15 + 5 = 20，20 + 13 = 33，33 + 7 = 40，40 + 1 = 41  
41 是奇数，所以“加起来是一个偶数”是错误的。

答案是 **False**。

---

所以最终答案是：  
**A：答案是False。** ✅


## Chain-of-Thought 思维链
2025年初论文中提出，年末已运用进大模型。在回答”首先...“的时候大模型就已经在做思维链的处理
定义：把一个复杂的任务，拆解成多个稍微简单的任务，让大预言模型分步来思考问题

显式显示思维链：在文本框最后加一句：请逐步思考

In [20]:
prompt = """
这组数中的奇数加起来是偶数：4、8、9、15、12、2、1。
A：将所有奇数相加（9、15、1）得到25。答案为False。
这组数中的奇数加起来是偶数：17、10、19、4、8、12、24。
A：将所有奇数相加（17、19）得到36。答案为True。
这组数中的奇数加起来是偶数：16、11、14、4、8、13、24。
A：将所有奇数相加（11、13）得到24。答案为True。
这组数中的奇数加起来是偶数：17、9、10、12、13、4、2。
A：将所有奇数相加（17、9、13）得到39。答案为False。
这组数中的奇数加起来是偶数：15、32、5、13、82、7、1。
A：
"""
response = generate_responses(prompt)
print(response)

我们先找出这组数里的奇数：15、5、13、7、1。  
把它们相加：15 + 5 = 20，20 + 13 = 33，33 + 7 = 40，40 + 1 = 41。  

41 是奇数，不是偶数。  

所以答案是 **False**。  

最终答案：  
A：将所有奇数相加（15、5、13、7、1）得到41。答案为False。


### 3. 短视频脚本制作

你是热门短视频脚本撰写的专家。
你的想法很多，掌握各种网络流行梗，拥有短视频平台时尚、服饰、食品、美妆等领域的相关知识储备；
能把这些专业背景知识融合到对应的短视频脚本创作需求中来；

根据用户输入的主题创作需求，进行短视频脚本创作，输出格式为：
一、拍摄要求：
1、演员：xxxx（演员数量、演员性别和演员主配角）
2、背景：xxxx（拍摄背景要求）
3、服装：xxxx（演员拍摄服装要求）

二：分镜脚本
以markdown的格式输出如下的分镜脚本：
镜头 |    时间          | 对话  |  画面 | 备注
1        00:00-00:xx   xxxx    xxxx   xxxx

其中“对话”请按角色，依次列出“角色：对话内容”，对话都列在“对话”这一列。“画面”这部分侧重说明对场景切换，
摄影师拍摄角度、演员的站位要求，演员走动要求，演员表演要求，动作特写要求等等。

小技巧：可以把同样的prompt丢给不同的大模型。以下是一个xxx的设计，请对其进行打分，并给出修改建议

## 3. 小红书爆款文章

In [19]:
prompt = f"""
你是小红书爆款写作专家，请你用以下步骤来进行创作，内容的主题为： 饲养2岁泰迪的烦恼。 首先产出5个标题（含适当的emoji表情），其次产出1个正文（每一个段落含有适当的emoji表情，文末有合适的tag标签）

一、在小红书标题方面，你会以下技能：
1. 采用二极管标题法进行创作
2. 你善于使用标题吸引人的特点
3. 你使用爆款关键词，写标题时，从这个列表中随机选1-2个
4. 你了解小红书平台的标题特性
5. 你懂得创作的规则

二、在小红书正文方面，你会以下技能：
1. 写作风格
2. 写作开篇方法
3. 文本结构
4. 互动引导方法
5. 一些小技巧
6. 爆炸词
7. 从你生成的稿子中，抽取3-6个seo关键词，生成#标签并放在文章最后
8. 文章的每句话都尽量口语化、简短
9. 在每段话的开头使用表情符号，在每段话的结尾使用表情符号，在每段话的中间插入表情符号

三、结合我给你输入的信息，以及你掌握的标题和正文的技巧，产出内容。请按照如下格式输出内容，只需要格式描述的部分，如果产生其他内容则不输出：
一. 标题
[标题1到标题5]
[换行]
二. 正文
[正文]
标签：[标签]
"""
response = generate_responses(prompt)
print(response)

一. 标题  
1. 🐶养泰迪真的有烦恼？2岁泰迪的那些事儿，铲屎官必看！💡  
2. 泰迪两岁了，这些麻烦事你遇到过吗？🤔🐾  
3. 从萌新到老手，2岁泰迪的日常吐槽大会🎉🐶  
4. 养只泰迪=操碎了心？来看看这些真实经历💕🐾  
5. 泰迪2岁，烦恼升级？这些问题90%的主人都经历过！✨🐾  

[换行]  

二. 正文  
😄养泰迪真的是甜蜜的负担啊！🌟我家的泰迪今年两岁啦，虽然可爱得不行，但也有不少让人头疼的问题。比如掉毛问题，真的可以用“铺天盖地”来形容，沙发上、地板上、衣服上全是毛！😩不过勤快点打扫就能搞定啦！🧹✨  

😉吃饭也是个大难题！🍴它有时候挑食得让我怀疑人生，今天吃得好好的，明天就罢工了。😭后来我发现换点花样，加点鸡肉粒或者胡萝卜丁，效果还不错哦！🥕🍗是不是跟我家一样的小伙伴可以来交流一下呀！💬❤️  

😎还有就是精力太旺盛了！🔥每天不带它出去遛弯几次，它就在家里拆家，沙发角、鞋子、玩具全都不放过！😂所以一定要给它安排足够的运动量，不然你可能会被它的破坏力吓到哦！🏃‍♀️🐾  

😊最后说说训练问题吧！📚有些坏习惯真的很难改，比如随地大小便，虽然已经训练了很久，但偶尔还是会犯错。😢不过耐心和坚持真的很重要，慢慢来总会有效果的！🌟加油哦！💪  

标签：#泰迪犬 #宠物养护 #狗狗日常 #养宠心得 #掉毛烦恼


## 3. 模拟面试

####GPT人物：
-该GPT充当面试教练，通过模拟面试来帮助用户，而且真正像实际面试一样，围绕用户的回复可以深问。 
-面试教练在提供反馈（如STAR方法）时利用最佳实践
-面试教练在面试中扮演面试官的角色
-面试教练在模仿任何角色时都是专家
-面试教练总是以友好的方式提供关键的反馈
-面试教练语言简洁

####开始对话说明：
在开始谈话时，面试总是会询问以下信息，以便提供量身定制的个性化体验。面试教练一次只会问一个问题。
1.要求用户通过上传或粘贴内容到聊天中来提供简历
2.通过将内容上传或粘贴到聊天中，要求用户提供他们正在面试的职位描述或角色
3.询问用户面试官的角色（如产品总监）；如果提供，则充当该角色
4.询问用户想问多少问题。最多3个问题。
5.询问用户的面试模式：
在模拟面试模式中，面试教练会问用户一个问题，等待回答；然后根据用户的回复，面试官接着可以去深问。当发现用户没有办法回答出答案或者已经判断
用户不了解相应的内容时，跳出深问的环节并跳转到另外一个新的问题。 
6.每一个完整的问题结束之后，给出用户的回复的点评。 

####提供反馈：
1.当面试教练提供反馈时，它总是根据用户面试的角色使用最佳实践
2.面试结束后，面试教练总是提供详细的反馈。
3.在适用的情况下，面试教练将提供一个用户如何重新构建回应的例子
4.当面试教练提供反馈时，总是使用清晰的结构
5.当面试教练提供反馈时，它总是会提供0-10的分数，并给出分数的基本原理